# Step 8 — CNN branch collapse investigation (DIAG-001)

**Frozen-checkpoint diagnostic only. No model training, weight updates, tuning, or official-test inference.**

Our paper-aligned SCV-IDS internal-validation run reached **55.87% macro-F1** (best checkpoint at epoch 41). Its saved diagnostic PCA showed **zero sample variance across the pooled 128 CNN features**, while the ViT and fusion remained variable. This notebook determines where that behavior originates.

### Scientific safeguards
- Verifies the original `scv_best.pt` against its published SHA-256.
- Recovers the **130.05 MiB original experiment archive** from the [draft GitHub Release](https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids/releases/tag/untagged-52b64328f8a128caaee7) when the old Kaggle working directory has disappeared. Download is authenticated with existing Kaggle secret `pushRare`, then checked byte-for-byte and each archived file verified.
- Uses **only** `UNSW_NB15_training-set.csv` to reconstruct the **saved original internal validation windows** using the **saved preprocessing vocabulary and scaler**; no new fitting and no official test access.
- Compares current checkpoint probabilities against frozen `validation_predictions.npz`, and CNN pooled vectors against frozen `embedding_vectors.npz`. **Fails closed** if they disagree.
- Saves layer/channel variance, zero activations, CNN/ViT attention and fusion statistics, inference-only zeroing controls, and controlled horizontal-flip sensitivity.
- Produces **12 figures × PNG/SVG/PDF** from versioned numeric CSVs, not fabricated visualizations. Saves source code, config, SHA hashes, sample-index digest and provenance.

**Interpretation:** zeroing an input branch is an out-of-distribution intervention, not definitive causal attribution. Horizontal flips are image perturbations, not proof of network adversarial robustness. All metrics belong to a *sample of the official training-derived validation set*, not the test partition.

**Kaggle settings:** Internet on; GPU on; attach `dsshayan/unsw-nb15`; authorize secret `pushRare`. Never expose token or upload row-level predictions to public GitHub.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json

REPO=Path("/kaggle/working/rare-attack-ids-step08")
URL="https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git"
BASE=Path("/kaggle/working/research_outputs")
ARCHIVE=Path("/kaggle/working/SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar")
OUTPUT=Path("/kaggle/working/research_outputs/SCV-CNN-DIAG-001")
TRAIN=Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")

assert TRAIN.is_file(), "Attach the original dsshayan/unsw-nb15 Kaggle input."
if REPO.exists():
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1",URL,str(REPO)],check=True)
print("Diagnostic research revision:",
      subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],
                              text=True).strip())

import torch
assert torch.cuda.is_available(), "Enable GPU in Kaggle accelerator settings."
subprocess.run(["nvidia-smi","--query-gpu=name,memory.total",
                "--format=csv,noheader"],check=True)

In [ ]:
# The verified archive is already backed up to a GitHub draft release.
# If this Kaggle session no longer contains the model, restore it, no training.
from kaggle_secrets import UserSecretsClient
token=UserSecretsClient().get_secret("pushRare")
env=os.environ.copy()
env["GITHUB_TOKEN"]=token
env["PYTHONPATH"]=str(REPO/"src")
env["OPENBLAS_NUM_THREADS"]="1"
env["MKL_NUM_THREADS"]="1"
env["OMP_NUM_THREADS"]="1"
try:
    subprocess.run([
        sys.executable,str(REPO/"scripts"/"restore_scv_release.py"),
        "--base",str(BASE),"--archive",str(ARCHIVE)
    ],cwd=str(REPO),env=env,check=True)
finally:
    env.pop("GITHUB_TOKEN",None)
    del token

In [ ]:
# Diagnostic forward passes on frozen internal validation, without training.
CONFIG=REPO/"configs"/"step08_cnn_diagnostic.json"
subprocess.run([
    sys.executable,str(REPO/"scripts"/"run_scv_cnn_diagnostic.py"),
    "--repo",str(REPO),"--config",str(CONFIG),
    "--base",str(BASE),"--output",str(OUTPUT)
],cwd=str(REPO),env=env,check=True)

m=json.loads((OUTPUT/"diagnostic_metrics.json").read_text())
print("Diagnostic windows:",m["n_diagnostic_validation_windows"])
print("Checkpoint replay max probability error:",
      m["prediction_replay_max_abs_probability_error"])
print("CNN pooled feature std:",m["current_cnn_mean_feature_std"])
print("Diagnostic figures:",m["figure_count"],"× 3 formats")

In [ ]:
# Secure synchronized upload of AGGREGATE CSVs and PNG/SVG/PDF figures only.
token=UserSecretsClient().get_secret("pushRare")
sync_env=env.copy()
sync_env["GITHUB_TOKEN"]=token
try:
    subprocess.run([
        sys.executable,str(REPO/"scripts"/"sync_scv_cnn_diagnostic.py"),
        "--run",str(OUTPUT)
    ],cwd=str(REPO),env=sync_env,check=True)
finally:
    sync_env.pop("GITHUB_TOKEN",None)
    del token

### What to return

Send the final line beginning **`VERIFIED CNN DIAGNOSTIC SYNC COMMIT:`**. If replay fails, share the error and do not rerun the 50 training epochs.

**Figure maintenance:** Regenerate all 12 scientific plots later with `python scripts/plot_scv_cnn_diagnostic.py --run OUTPUT_DIR`. Style edits require no checkpoint and no new inference. Only the original, separately preserved archive contains full checkpoint weights and per-row validation predictions.